# RAG Misinformation Experiments — Results (P5: In RAG We Trust?)

This notebook loads `results/experiment_results.csv` (produced by
`python -m src.evaluator`) and charts the three experiments **per dataset**
(synthetic, HotpotQA, FEVER) and, when present, per model and retrieval setting.

- **Exp 1**: real evidence only (baseline)
- **Exp 2**: real + poisoned, no verification
- **Exp 3**: real + poisoned, with verification

We expect the **hallucination rate** to rise from Exp 1 → Exp 2, then fall
again in Exp 3 if the verification (meta-)prompt helps.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the results file whether run from the repo root or from notebooks/.
CSV_PATH = "../results/experiment_results.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "results/experiment_results.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print("datasets:", sorted(df['dataset'].unique()))
print("models:  ", sorted(df['model'].unique()))
print("top_k:   ", sorted(df['top_k'].unique()))
df.head()

## Aggregate metrics per (dataset, experiment)

`accuracy` is the mean of `is_correct` over all items; `hallucination_rate` is
the mean of `is_hallucinated` over poisoned-target items only.

In [ ]:
EXP_ORDER = ["exp1_baseline", "exp2_poisoned", "exp3_poisoned_verify"]
EXP_LABELS = {
    "exp1_baseline": "Exp1\nbaseline",
    "exp2_poisoned": "Exp2\npoisoned",
    "exp3_poisoned_verify": "Exp3\n+verify",
}

accuracy = df.groupby(["dataset", "experiment"])["is_correct"].mean()
poisoned = df[df["is_poisoned_target"] == True]
hallucination = poisoned.groupby(["dataset", "experiment"])["is_hallucinated"].mean()

summary = pd.DataFrame({"accuracy": accuracy, "hallucination_rate": hallucination})
summary = summary.reset_index()
summary["experiment"] = pd.Categorical(summary["experiment"], EXP_ORDER, ordered=True)
summary = summary.sort_values(["dataset", "experiment"]).reset_index(drop=True)
summary

## Hallucination rate per experiment, faceted by dataset

The core P5 figure: did the poison win, and did verification claw it back?

In [ ]:
datasets = sorted(df["dataset"].unique())
fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    sub = summary[summary["dataset"] == ds].set_index("experiment").reindex(EXP_ORDER)
    labels = [EXP_LABELS[e] for e in EXP_ORDER]
    ax.bar(labels, sub["hallucination_rate"], color=["#4c72b0", "#c44e52", "#55a868"])
    ax.set_title(ds)
    ax.set_ylim(0, 1)
    for i, v in enumerate(sub["hallucination_rate"].fillna(0)):
        ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
axes[0].set_ylabel("Hallucination rate")
fig.suptitle("How often the model repeated the poisoned (fake) answer")
plt.tight_layout()
plt.show()

## Accuracy vs hallucination, side by side per dataset

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4), sharey=True)
if len(datasets) == 1:
    axes = [axes]
width = 0.35
x = np.arange(len(EXP_ORDER))

for ax, ds in zip(axes, datasets):
    sub = summary[summary["dataset"] == ds].set_index("experiment").reindex(EXP_ORDER)
    ax.bar(x - width/2, sub["accuracy"].fillna(0), width, label="accuracy", color="#55a868")
    ax.bar(x + width/2, sub["hallucination_rate"].fillna(0), width, label="hallucination", color="#c44e52")
    ax.set_xticks(x)
    ax.set_xticklabels([EXP_LABELS[e] for e in EXP_ORDER])
    ax.set_ylim(0, 1)
    ax.set_title(ds)
axes[0].set_ylabel("rate")
axes[-1].legend()
fig.suptitle("Accuracy vs hallucination rate by experiment")
plt.tight_layout()
plt.show()

## Compare models / retrieval settings (if swept)

P5 asks for the metrics *across models and retrieval settings*. If the run
swept more than one model or `top_k`, this shows hallucination rate broken down
by model for the poisoned experiment.

In [ ]:
pois2 = df[(df["is_poisoned_target"] == True) & (df["experiment"] == "exp2_poisoned")]
by_model = pois2.groupby(["dataset", "model"])["is_hallucinated"].mean().unstack("model")
if by_model.shape[1] >= 1:
    by_model.plot(kind="bar", figsize=(8, 4), ylim=(0, 1))
    plt.ylabel("Hallucination rate (Exp 2)")
    plt.title("Poisoned-condition hallucination rate by model")
    plt.tight_layout()
    plt.show()
by_model

## Self-consistency (if the SC pass was enabled)

Populated only when the evaluator is run with `self_consistency_samples > 0`.
Higher = the model gives the same answer more reliably when resampled.

In [ ]:
sc = df.dropna(subset=["self_consistency"])
sc = sc[sc["self_consistency"] != ""]
if len(sc):
    sc_agg = sc.groupby(["dataset", "experiment"])["self_consistency"].mean().unstack("experiment")
    sc_agg = sc_agg.reindex(columns=EXP_ORDER)
    sc_agg.plot(kind="bar", figsize=(8, 4), ylim=(0, 1))
    plt.ylabel("Self-consistency")
    plt.title("Answer self-consistency by dataset and experiment")
    plt.tight_layout()
    plt.show()
else:
    print("No self-consistency data — run the evaluator with self_consistency_samples > 0.")

## Inspect individual answers

Useful for the report — see exactly what the model said in each condition.

In [ ]:
cols = ["dataset", "experiment", "model", "question", "model_answer",
        "is_correct", "is_hallucinated", "poison_doc_retrieved"]
pd.set_option("display.max_colwidth", 80)
df[cols]